In [ ]:
# %% Deep learning - Section 22.206
#    Code challenge 35: Style transfer with Alexnet
#
#    1) Start from code from video 22.205 (style transfer with VGG-19)
#    2) Use AlexNet instead of VGG-19
#    3) Pick your own content and style image
#    4) Explore the meta-parameters (layers, weights for layers and losses)
#    5) HAve fun !

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Import and freeze VGG-19

# Import the model
alex_net = torchvision.models.alexnet(weights=True)

# Freeze all the layers
for p in alex_net.parameters():
    p.requires_grad = False

# Switch to evaluation mode
alex_net.eval()


In [ ]:
# %% Use GPU and ship model there

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

alex_net.to(device)


In [ ]:
# %% Import images

uploaded = files.upload()

for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))


In [ ]:
# %% Load images

img_content = imageio.v2.imread('content_image_17.jpg')
img_style   = imageio.v2.imread('style_image_12.jpg')

# Initialize the target image as random numbers
img_target = np.random.randint(low=0,high=255,size=img_content.shape,dtype=np.uint8)
# img_target = img_content

# Check sizes
print("Content img:")
print(img_content.shape)
print("\nTarget img:")
print(img_target.shape)
print("\nStyle img:")
print(img_style.shape)


In [ ]:
# %% Reduce image sizes to match the model (also, would take too long)

# Transforms (resize and normalise)
m  = [0.485, 0.456, 0.406]
s  = [0.229, 0.224, 0.225]

Ts = T.Compose([ T.ToTensor(),
                 T.Resize(256),
                 T.Normalize(mean=m,std=s) ])

# Apply transforms (unsqueeze to turn into 4D tensor and ship to GPU)
img_content = Ts( img_content ).unsqueeze(0).to(device)
img_style   = Ts( img_style   ).unsqueeze(0).to(device)
img_target  = Ts( img_target  ).unsqueeze(0).to(device)

# Check sizes
print("Content img:")
print(img_content.shape)
print("\nTarget img:")
print(img_target.shape)
print("\nStyle img:")
print(img_style.shape)


In [ ]:
# %% Plotting

# Plot the "before" pics
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(phi*10,10))

pic = img_content.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[0].imshow(pic)
ax[0].set_title('Content picture')

pic = img_target.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[1].imshow(pic)
ax[1].set_title('Target picture')

pic = img_style.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[2].imshow(pic)
ax[2].set_title('Style picture')

plt.savefig('figure68_code_challenge_35.png')
plt.show()
files.download('figure68_code_challenge_35.png')


In [69]:
# Function to returns feature maps

def get_feature_map_activations(img,net):

    # Prealloacte feature maps lists
    feature_maps  = []
    feature_names = []

    conv_layer_idx = 0

    # Loop through all layers in the features block (the conv block)
    for layer_num in range(len(net.features)):

        # Print out info from this layer
        # print(layer_num,net.features[layer_num])

        # Process the image through this layer
        img = net.features[layer_num](img)

        # Store the image if it is a conv2d layer
        if 'Conv2d' in str(net.features[layer_num]):
            feature_maps.append( img )
            feature_names.append( 'Conv_layer_' + str(conv_layer_idx) )
            conv_layer_idx += 1

    return feature_maps,feature_names


In [70]:
# Function to compute the Gram matrix of the feature activation map

def gram_matrix(M):

    # Reshape to 2D
    _,chans,height,width = M.shape
    M = M.reshape(chans,height*width)

    # Compute covariance matrix (scale by 1/n)
    gram = torch.mm(M,M.t()) / (chans*height*width)

    return gram


In [ ]:
# Inspect the output of the feature maps function

feat_maps,feat_names = get_feature_map_activations(img_content,alex_net)

# Print some info
for i in range(len(feat_names)):
    print('Feature map "%s" is size %s'%(feat_names[i],(feat_maps[i].shape)))


In [176]:
# %% Pick meta-parameters for the transfer

# Layers to use (salisbury)
layers4content = [ 'Conv_layer_0','Conv_layer_2' ]
layers4style   = [ 'Conv_layer_0','Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4' ]
weights4style  = [      .8       ,     .7      ,     .5      ,     .1      ,     .1       ]

# Layers to use (amiens et parthenon)
layers4content = [ 'Conv_layer_1' ]
layers4style   = [ 'Conv_layer_0','Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4' ]
weights4style  = [      .1       ,     .1      ,     .1     ,      .01      ,       .01       ]


In [177]:
# %% Run the style transfer

# Copy the target image and ship to GPU
target = img_target.clone()
target.requires_grad = True

target = target.to(device)
style_scaling = 1e6

# Content and style feature maps
content_feature_maps,content_feature_names = get_feature_map_activations(img_content,alex_net)
style_feature_maps,style_feature_names     = get_feature_map_activations(img_style,alex_net)

# Number training epochs
numepochs = 1500

# Optimizer for backpropagation (RMSprop is the recommended one)
optimizer = torch.optim.RMSprop([target],lr=.001)

losses_style   = []
losses_content = []
frames         = []

for epoch_i in range(numepochs):

    # Save initial frame
    if epoch_i == 0:
        frames.append(target.detach().cpu().clone())

    # Get the target feature maps
    target_feature_maps,target_feature_names = get_feature_map_activations(target,alex_net)

    # Preallocate individual loss components as tensors to ensure combined_loss is also a tensor
    style_loss   = torch.tensor(0.0,device=device)
    content_loss = torch.tensor(0.0,device=device)

    # Loop over layers
    for layer_i in range(len(target_feature_names)):

        # compute the content loss (MSE)
        if target_feature_names[layer_i] in layers4content:
            content_loss += torch.mean((target_feature_maps[layer_i] - content_feature_maps[layer_i])**2)

        # Compute the style loss (MSE)
        if target_feature_names[layer_i] in layers4style:

            # Gram matrices
            G_target = gram_matrix(target_feature_maps[layer_i])
            G_style  = gram_matrix(style_feature_maps[layer_i])

            # Compute loss (de-weighted with increasing depth)
            style_loss += torch.mean((G_target - G_style)**2) * weights4style[layers4style.index(target_feature_names[layer_i])]

    # Store separate losses
    losses_style.append(style_loss.item())
    losses_content.append(content_loss.item())

    # Combine losses (scale style losses)
    combined_loss = style_scaling*style_loss + content_loss

    # Backpropagation
    optimizer.zero_grad()
    combined_loss.backward()
    optimizer.step()

    # Store every 100 epochs
    if epoch_i % 100 == 0:
        frames.append(target.detach().cpu().clone())

# Save final frame
frames.append(target.detach().cpu().clone())


In [ ]:
# %% Plotting

# Plot the "after" pics
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*10,10))

gs = GridSpec(1,3,width_ratios=[1,1,0.6])
ax = [fig.add_subplot(gs[i]) for i in range(3)]

pic = img_content.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[0].imshow(pic)
ax[0].set_title('Content picture',fontweight='bold')
ax[0].set_xticks([])
ax[0].set_yticks([])

pic = torch.sigmoid(target).cpu().detach().squeeze().numpy().transpose((1,2,0))
ax[1].imshow(pic)
ax[1].set_title('Target picture',fontweight='bold')
ax[1].set_xticks([])
ax[1].set_yticks([])

pic = img_style.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[2].imshow(pic)
ax[2].set_title('Style picture',fontweight='bold')
ax[2].set_xticks([])
ax[2].set_yticks([])

plt.savefig('figure69_code_challenge_35.png')
plt.show()
files.download('figure69_code_challenge_35.png')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*10,10))

upsampled = F.interpolate(torch.sigmoid(target),scale_factor=12,mode='bilinear',align_corners=False)
pic       = upsampled.cpu().detach().squeeze().numpy().transpose((1,2,0))

plt.imshow(pic)
plt.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

plt.savefig('figure70_code_challenge_35.png',bbox_inches='tight',pad_inches=0)
plt.show()
files.download('figure70_code_challenge_35.png')


In [ ]:
# %% Clip

import matplotlib.animation as animation

# Produce movie
def tensor_to_image(t):
    img = torch.sigmoid(t).squeeze().permute(1,2,0).numpy()
    return img

frames_np = [tensor_to_image(f) for f in frames]

# Generate clip
phi = (1 + np.sqrt(5)) / 2
fig, ax = plt.subplots(figsize=(phi*6,6))
ax.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

im = ax.imshow(frames_np[0])

def update(i):
    im.set_array(frames_np[i])
    return (im,)

# Interval is ms between frames
animat = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames_np),
    interval=200 )

animat.save('figure71_code_challenge_35_video.gif',writer='pillow',fps=5)
files.download('figure71_code_challenge_35_video.gif')

animat.save('figure72_code_challenge_35_video.mp4', fps=5)
files.download('figure72_code_challenge_35_video.mp4')
